In [ ]:
import platform
import psutil
from typing import Tuple, Union
from timeit import timeit
from typing import Callable
from warnings import warn

# PyTorch dependencies
import torch
import torch.backends.opt_einsum as opt_einsum
from torch.func import jacfwd
from torch import Tensor


# Internal dependencies
from thoad import backward, Controller

In [141]:
# control size of tensors
TENSOR_SCALE: Union[int, float] = 1
REPEAT_SCALE: Union[int, float] = 1

In [142]:
sys: platform.uname_result = platform.uname()
print(f"system           {sys.system} {sys.release} {sys.version}")

system           Windows 11 10.0.26100


In [143]:
torch_dev: torch.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch_dev.type == 'cuda':
    idx = torch_dev.index if torch_dev.index is not None else 0
    props: "_CudaDeviceProperties" = torch.cuda.get_device_properties(idx)
    name: str = props.name
    total_mem_gb: float = props.total_memory / (1024**3)
    print(f"using device     {torch_dev} -> {name}")
    print(f"device memory    {total_mem_gb:.1f} GB)")
else:
    cpu_name: str = platform.processor() or "CPU"
    print(f"using device     {torch_dev} -> {cpu_name}")
    print(f"physical cores   {psutil.cpu_count(logical=False)}")
    print(f"logical cores    {psutil.cpu_count(logical=True)}")

using device     cpu -> AMD64 Family 23 Model 160 Stepping 0, AuthenticAMD
physical cores   4
logical cores    8


In [144]:
if opt_einsum.is_available():
    opt_einsum.enabled = True
    print("opt_einsum backend enabled")
    opt_einsum.strategy = "optimal"
else:
    warn(
        "opt_einsum backend is not available. "
        "For better performance, install and enable opt_einsum.",
        UserWarning
    )

opt_einsum backend enabled


## **Benchmark differentiations on full MLP**

definition of MLP

In [145]:
def torch_foward_pass(X: Tensor, *params) -> Tensor:
    T: Tensor = X
    for i, P in enumerate(params):
        last_step: bool = i == (len(params) - 1)
        T = T @ P
        T = torch.softmax(T, dim=1) if last_step else torch.relu(T)
    return T

definition of helper functions to meassure differentiation times

In [ ]:
def compose_jacfwd(fn: Callable, n: int) -> Callable:
    g: Callable = fn
    for _ in range(n):
        g = jacfwd(g)
    return g


def time_func_differentiation(reps: int, order:int, X: Tensor, *params) -> float:
    X.requires_grad_(True)
    params: list[Tensor] = [P.requires_grad_(False) for P in params]
    assert all(not param.requires_grad for param in params)
    def _fixed_forward(X: Tensor) -> Tensor:
        Y: Tensor = torch_foward_pass(X, *params)
        return Y
    differentiator: Callable = compose_jacfwd(fn=_fixed_forward, n=order)
    def _foward_and_backward() -> None:
        differentiator(X)
    time: float = timeit(
        lambda: _foward_and_backward(),
        number=reps,
    )
    return time


def time_thoad_differentiation(reps: int, order:int, X: Tensor, *params) -> float:
    X.requires_grad_(True)
    params: list[Tensor] = [P.requires_grad_(False) for P in params]
    assert all(not param.requires_grad for param in params)
    def _foward_and_backward() -> None:
        T: Tensor = torch_foward_pass(X, *params)
        ctrl: Controller = backward(
            tensor=T,
            order=order,
            crossings=False,
            keep_batch=True,
        )
        ctrl.clear()
        return None
    time: float = timeit(
        lambda: _foward_and_backward(),
        number=reps,
    )
    return time

differentiation computational cost w.r.t. **order** and **batch size**

In [ ]:
for o in [1, 2]:
    print(f"\nORDER {o}")
    for batch_size in [15, 30, 45, 60, 75, 90]:
        batch_size //= o
        param_size: int = int(5 * TENSOR_SCALE)
        x_shape: Tuple[int, int] = (batch_size, param_size)
        p_shape: Tuple[int, int] = (param_size, param_size)

        # create torch tensors
        torch_X: Tensor = torch.rand(size=x_shape, device=torch_dev)
        torch_params: list[Tensor] = [
            torch.rand(size=p_shape, device=torch_dev) for _ in range(3)
        ]

        reps: int = int(600 * (1 / batch_size) * (1/o) * REPEAT_SCALE)
        func_time: float = time_func_differentiation(reps, o, torch_X, *torch_params)
        thoad_time: float = time_thoad_differentiation(reps, o, torch_X, *torch_params)
        print(
            f"batch size: {batch_size:02d} -> "
            f"func time: {func_time/reps:.4f}  thoad time: {thoad_time/reps:.4f}"
        )

#



ORDER 1
batch size: 15 -> func time: 0.0055  thoad time: 0.0193
batch size: 30 -> func time: 0.0030  thoad time: 0.0188
batch size: 45 -> func time: 0.0044  thoad time: 0.0169
batch size: 60 -> func time: 0.0055  thoad time: 0.0196
batch size: 75 -> func time: 0.0086  thoad time: 0.0246
batch size: 90 -> func time: 0.0075  thoad time: 0.0216

ORDER 2
batch size: 07 -> func time: 0.0097  thoad time: 0.0290
batch size: 15 -> func time: 0.0217  thoad time: 0.0867
batch size: 22 -> func time: 0.0489  thoad time: 0.0243
batch size: 30 -> func time: 0.1034  thoad time: 0.0270
batch size: 37 -> func time: 0.1836  thoad time: 0.0316
batch size: 45 -> func time: 0.3026  thoad time: 0.0424


differentiation computational cost w.r.t. **order** and **param size**

In [148]:
for o in [1, 2]:
    print(f"\nORDER {o}")
    for param_size in [10, 20, 30, 40, 50, 60]:
        batch_size: int = int(5 * TENSOR_SCALE)
        param_size //= o
        x_shape: Tuple[int, int] = (batch_size, param_size)
        p_shape: Tuple[int, int] = (param_size, param_size)

        # create torch tensors
        torch_X: Tensor = torch.rand(size=x_shape, device=torch_dev)
        torch_params: list[Tensor] = [
            torch.rand(size=p_shape, device=torch_dev) for _ in range(3)
        ]

        reps: int = int(600 * (1 / param_size) * (1/o) * REPEAT_SCALE)
        func_time: float = time_func_differentiation(reps, o, torch_X, *torch_params)
        thoad_time: float = time_thoad_differentiation(reps, o, torch_X, *torch_params)
        print(
            f"param size: {param_size:02d} -> "
            f"func time: {func_time/reps:.4f}  thoad time: {thoad_time/reps:.4f}"
        )




ORDER 1
param size: 10 -> func time: 0.0020  thoad time: 0.0099
param size: 20 -> func time: 0.0014  thoad time: 0.0105
param size: 30 -> func time: 0.0018  thoad time: 0.0105
param size: 40 -> func time: 0.0053  thoad time: 0.0133
param size: 50 -> func time: 0.0022  thoad time: 0.0100
param size: 60 -> func time: 0.0024  thoad time: 0.0108

ORDER 2
param size: 05 -> func time: 0.0052  thoad time: 0.0168
param size: 10 -> func time: 0.0076  thoad time: 0.0177
param size: 15 -> func time: 0.0120  thoad time: 0.0215
param size: 20 -> func time: 0.0224  thoad time: 0.0214
param size: 25 -> func time: 0.0408  thoad time: 0.0277
param size: 30 -> func time: 0.0609  thoad time: 0.0378


differentiation computational cost w.r.t. **graph depth** (param gradients included)

In [149]:
for o in [1, 2]:
    print(f"\nORDER {o}")
    for depth in [2, 3, 4, 5, 6, 7, 8]:
        batch_size: int = 40 // o
        param_size: int = int(10 // o * TENSOR_SCALE)
        x_shape: Tuple[int, int] = (batch_size, param_size)
        p_shape: Tuple[int, int] = (param_size, param_size)

        # create torch tensors
        torch_X: Tensor = torch.rand(size=x_shape, device=torch_dev)
        torch_params: list[Tensor] = [
            torch.rand(size=p_shape, device=torch_dev) for _ in range(depth)
        ]

        reps: int = int(600 * (1 / depth) * (1/o) * REPEAT_SCALE)
        func_time: float = time_func_differentiation(reps, o, torch_X, *torch_params)
        thoad_time: float = time_thoad_differentiation(reps, o, torch_X, *torch_params)
        print(
            f"depth size: {depth:02d} -> "
            f"func time: {func_time/reps:.4f}  thoad time: {thoad_time/reps:.4f}"
        )


ORDER 1
depth size: 02 -> func time: 0.0036  thoad time: 0.0110
depth size: 03 -> func time: 0.0044  thoad time: 0.0134
depth size: 04 -> func time: 0.0059  thoad time: 0.0161
depth size: 05 -> func time: 0.0068  thoad time: 0.0198
depth size: 06 -> func time: 0.0078  thoad time: 0.0224
depth size: 07 -> func time: 0.0098  thoad time: 0.0275
depth size: 08 -> func time: 0.0112  thoad time: 0.0317

ORDER 2
depth size: 02 -> func time: 0.0232  thoad time: 0.0151
depth size: 03 -> func time: 0.0325  thoad time: 0.0316
depth size: 04 -> func time: 0.0569  thoad time: 0.0382
depth size: 05 -> func time: 0.0694  thoad time: 0.0469
depth size: 06 -> func time: 0.0823  thoad time: 0.0539
depth size: 07 -> func time: 0.0941  thoad time: 0.0608
depth size: 08 -> func time: 0.1061  thoad time: 0.0689
